## 環境設置

In [1]:
try:
    import numpy as np  # noqa: F401
except ImportError:
    import sys
    !{sys.executable} -m pip install -q numpy

# Case-02.5:反曲點法 + 虛功法——另一個「分析端」的快速手算工具

依討論後的規劃,這是繼位移法(Case-02 精確閉合解)之後,第二個
獨立的分析方法。跟 Case-02 用完全相同的幾何/斷面/載重,直接對照
誰準、誰快、差多少。

**反曲點法本身只用靜力平衡,完全不用 EI,算不出位移**——這裡額外
加一步虛功法(單位荷重法),拿反曲點法算出的彎矩圖去積分求位移,
這是結構學裡標準的銜接技巧,不是反曲點法自己內建的功能。

**這不是要取代 Case-02 的精確解**,是要量化「用這個更快的近似法,
要付出多少準確度代價」,而且這個代價本身應該跟 Case-04.5 發現的
$k_b/k_c$ 有關——這裡直接驗證這個猜測對不對。

## 第 1 課:沿用 Case-02 完全相同的問題設定

In [2]:
h, L = 3.5, 6.0
E = 25_000_000.0
Ic, Ib = 0.0010, 0.0015
F = 50.0   # kN, 總側向力(對稱分攤於兩屋頂節點, 呼應Case-02的施力方式)

print(f"h={h}m, L={L}m, E={E:.3e} kN/m^2, Ic={Ic}m^4, Ib={Ib}m^4, F={F}kN")

h=3.5m, L=6.0m, E=2.500e+07 kN/m^2, Ic=0.001m^4, Ib=0.0015m^4, F=50.0kN


## 第 2 課:反曲點法——柱端彎矩(純靜力平衡)

單跨單層構架,兩根柱都是外柱(沒有內柱 2 倍剪力的規則,那個規則
只在多柱情況下才會用到,回頭在 Case-04 的多跨模型會再用上)。反曲
點假設在柱中點、梁中點。

In [3]:
V_col = F/2   # 兩柱皆外柱, 均分
M_col = V_col * (h/2)   # 反曲點在柱中點, 柱端彎矩 = 柱剪力 x 半柱高
M_beam = M_col   # 節點力矩平衡: 單一節點只有一柱一梁交會, 梁端彎矩=柱端彎矩

print(f"柱剪力 V_col = {V_col} kN")
print(f"柱端彎矩 M_col = {M_col} kN-m(反曲點在柱中點)")
print(f"梁端彎矩 M_beam = {M_beam} kN-m(節點力矩平衡)")

柱剪力 V_col = 25.0 kN
柱端彎矩 M_col = 43.75 kN-m(反曲點在柱中點)
梁端彎矩 M_beam = 43.75 kN-m(節點力矩平衡)


## 第 3 課:虛功法求位移

拿第 2 課的彎矩圖,套用虛功法(單位荷重法)。因為虛擬單位力施加
位置/方向跟真實力完全相同,虛擬彎矩圖 = 真實彎矩圖 ÷ F,積分可以
化簡成 $\Delta = \frac{1}{F}\sum\int \frac{M^2}{EI}dx$。

柱跟梁的彎矩圖都是「從反曲點(M=0)線性變化到端點(M=M_max)」的
三角形分布,這種分布的 $\int M^2 dx$ 有現成公式:$M_{max}^2 \times \ell/3$
(ℓ為該段長度)。

In [4]:
# 柱: 上下半柱對稱(雙曲率), 每半柱長h/2
int_col = M_col**2 * (h/2) / 3 * 2 * 2   # x2(上下兩半) x2(兩根柱)

# 梁: 左右半梁對稱, 每半梁長L/2
int_beam = M_beam**2 * (L/2) / 3 * 2   # x2(左右兩半, 只有一根梁)

print(f"柱段 integral(M^2 dx) = {int_col:.4f} kN^2-m^3")
print(f"梁段 integral(M^2 dx) = {int_beam:.4f} kN^2-m^3")

Delta_portal = (1/F) * (int_col/(E*Ic) + int_beam/(E*Ib))
print(f"\n反曲點法+虛功法 位移 Delta = {Delta_portal:.8f} m")

柱段 integral(M^2 dx) = 4466.1458 kN^2-m^3
梁段 integral(M^2 dx) = 3828.1250 kN^2-m^3

反曲點法+虛功法 位移 Delta = 0.00561458 m


## 第 4 課:與 Case-02 精確解對照

Case-02 用位移法(slope-deflection)推導的精確閉合解:
Delta=0.00528792m,柱端彎矩=50.750kN-m。

In [5]:
Delta_exact = 0.00528792
M_exact = 50.750

m_err = abs(M_col-M_exact)/M_exact
d_err = abs(Delta_portal-Delta_exact)/Delta_exact

print(f"柱端彎矩: 反曲點法={M_col} kN-m, 精確解={M_exact} kN-m, 誤差={m_err:.2%}")
print(f"位移:     反曲點法+虛功法={Delta_portal:.6f} m, 精確解={Delta_exact} m, 誤差={d_err:.2%}")

kb_over_kc = (E*Ib/L)/(E*Ic/h)
print(f"\nkb/kc = {kb_over_kc:.3f}")

柱端彎矩: 反曲點法=43.75 kN-m, 精確解=50.75 kN-m, 誤差=13.79%
位移:     反曲點法+虛功法=0.005615 m, 精確解=0.00528792 m, 誤差=6.18%

kb/kc = 0.875


## 第 5 課:掃描 $k_b/k_c$——驗證跟 Case-04.5 同一套規律

固定梁斷面,改變柱斷面,看反曲點法的誤差怎麼隨 $k_b/k_c$ 變化。

In [6]:
def portal_plus_virtual_work(Ic_test):
    V_col = F/2
    M_col = V_col*(h/2)
    M_beam = M_col
    int_col = M_col**2*(h/2)/3 * 2*2
    int_beam = M_beam**2*(L/2)/3 * 2
    Delta = (1/F)*(int_col/(E*Ic_test) + int_beam/(E*Ib))
    kb_kc = (E*Ib/L)/(E*Ic_test/h)
    return M_col, Delta, kb_kc   # 注意: M_col跟kb_kc有關(見下方說明), 這裡M_col不隨Ic變, 但kb_kc會變


def exact_slope_deflection(Ic_test):
    kc = E*Ic_test/h
    kb = E*Ib/L
    K = 12*E*Ic_test/h**3 * (kc+6*kb)/(2*kc+3*kb)
    Delta = F/K
    psi = Delta/h
    theta = 3*psi*kc/(2*kc+3*kb)
    M_base = abs(2*E*Ic_test/h*(theta-3*psi))
    return M_base, Delta


print(f"{'Ic':<10}{'kb/kc':<10}{'M誤差':<10}{'Delta誤差':<10}")
test_results = []
for Ic_test in [0.0010, 0.0005, 0.0002, 0.0001]:
    M_p, D_p, kbkc = portal_plus_virtual_work(Ic_test)
    M_e, D_e = exact_slope_deflection(Ic_test)
    m_err = abs(M_p-M_e)/M_e
    d_err = abs(D_p-D_e)/D_e
    test_results.append((Ic_test, kbkc, m_err, d_err))
    print(f"{Ic_test:<10}{kbkc:<10.2f}{m_err:<10.1%}{d_err:<10.1%}")

# 驗證誤差隨kb/kc增加而遞減(趨勢方向要對)
kbkc_vals = [r[1] for r in test_results]
d_errs = [r[3] for r in test_results]
assert all(kbkc_vals[i] < kbkc_vals[i+1] for i in range(len(kbkc_vals)-1)), "kb/kc應該遞增"
assert all(d_errs[i] > d_errs[i+1] for i in range(len(d_errs)-1)), "位移誤差應該隨kb/kc增加而遞減"
print("\n[PASS] 誤差隨kb/kc增加而遞減, 與Case-04.5發現的規律方向一致")

Ic        kb/kc     M誤差       Delta誤差   
0.001     0.88      13.8%     6.2%      
0.0005    1.75      8.0%      2.0%      
0.0002    4.38      3.5%      0.4%      
0.0001    8.75      1.8%      0.1%      

[PASS] 誤差隨kb/kc增加而遞減, 與Case-04.5發現的規律方向一致


## 總結表

In [7]:
print("="*55)
print("Case-02.5 反曲點法+虛功法結果總結")
print("="*55)
print(f"{'柱端彎矩(反曲點法)':<24}{M_col} kN-m")
print(f"{'柱端彎矩(Case-02精確解)':<24}{M_exact} kN-m")
print(f"{'彎矩誤差':<24}{m_err:.1%}")
print(f"{'位移(反曲點法+虛功法)':<24}{Delta_portal:.6f} m")
print(f"{'位移(Case-02精確解)':<24}{Delta_exact} m")
print(f"{'位移誤差':<24}{d_err:.1%}")
print(f"{'本案例kb/kc':<24}{kb_over_kc:.3f}")
print()
print("Case-02.5 [PASS] -- 反曲點法的準確度規律與Case-04.5一致")
print("(kb/kc大->柱軟梁硬->誤差小; kb/kc小->柱硬梁軟->誤差大)")

Case-02.5 反曲點法+虛功法結果總結
柱端彎矩(反曲點法)              43.75 kN-m
柱端彎矩(Case-02精確解)        50.75 kN-m
彎矩誤差                    1.8%
位移(反曲點法+虛功法)            0.005615 m
位移(Case-02精確解)          0.00528792 m
位移誤差                    0.1%
本案例kb/kc                0.875

Case-02.5 [PASS] -- 反曲點法的準確度規律與Case-04.5一致
(kb/kc大->柱軟梁硬->誤差小; kb/kc小->柱硬梁軟->誤差大)
